# 01 · Data ingestion

The source file becomes two things in this notebook, and they are not copies of
each other.

| Destination | Engine | Why it exists |
|---|---|---|
| `bank_churn.customers_raw` | Lakebase (Postgres) | The landing table. Faithful, unconstrained, auditable |
| `bank_churn.customers` | Lakebase (Postgres) | Typed and constrained. Foreign-key target for predictions |
| `bank_churn_eng.bronze.customers_raw` | Delta | The analytical copy, with history and time travel |

Postgres receives it because the retention team works row by row and needs
foreign keys. Delta receives it because the modelling work scans columns and
needs versions. Writing to one and not the other would break something later.

Before either write happens, eight checks run against the file. The point of
running them **before** loading is that a failure becomes a documented finding
instead of a constraint violation with no context.

In [2]:
import sys
sys.path.append("../src")

from config import bootstrap

ctx = bootstrap()
spark, w = ctx.spark, ctx.w

connected to Databricks
  branch   : sandbox
  catalog  : bank_churn_eng
  identity : juzoushio@...


## 1 · The source

The CSV lives in a Unity Catalog volume rather than being downloaded at run time.
That is a deliberate difference: the notebook reproduces without Kaggle
credentials, and the file sits under the same governance as the tables.

The path `/Volumes/...` **only exists on Databricks compute**. From this local
process `os.path.exists` on it returns `False`, because the filesystem is remote.
It has to be reached through Spark or through the Files API.

Two things are read, and each one goes where it belongs:

- The **hash** is computed on the cluster, from the binary content. The file never
  travels for this.
- The **rows** do come down to the driver, because the load into Postgres runs
  through `pg8000`, a plain Python driver. `toPandas()` marks that boundary
  explicitly instead of letting it happen quietly inside a `read_csv`.

In [3]:
from pyspark.sql.functions import sha2, col
from config import UC_CATALOG, VOLUME_PATH

files = spark.sql(f"LIST '/Volumes/{UC_CATALOG}/bronze/raw_data/'").toPandas()
print(files[["name", "size"]].to_string(index=False))

     name   size
churn.csv 684858


In [4]:
# Hash on the cluster: the file does not move for this.
meta = (spark.read.format("binaryFile").load(VOLUME_PATH)
        .select(sha2(col("content"), 256).alias("sha256"), col("length"))
        .first())
sha256 = meta["sha256"]

# The rows do come down. header and inferSchema are both required: without the
# first, the header row is read as data and every column arrives as _c0, _c1...
raw = (spark.read
       .option("header", True)
       .option("inferSchema", True)
       .csv(VOLUME_PATH)
       .toPandas())

print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head(3)

10,000 rows x 14 columns


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1


## 2 · Provenance

Seven fields that answer, months from now, "where exactly did this come from".
The `sha256` is the one that matters: it identifies the file itself, so if the
source is ever replaced the change is detectable rather than invisible.

`source_url` is kept even though nothing is downloaded. The file did come from
Kaggle originally, and that fact does not stop being true because the copy now
lives in a volume.

In [5]:
import datetime as dt
from config import SOURCE_URL, BRANCH_NAME

TRACE = {
    "source_url":  SOURCE_URL,
    "source_file": VOLUME_PATH,
    "read_at":     dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "size_bytes":  meta["length"],
    "rows":        int(raw.shape[0]),
    "columns":     int(raw.shape[1]),
    "sha256":      sha256,
    "branch":      BRANCH_NAME,
}

for k, v in TRACE.items():
    print(f"{k:>13}: {v}")

   source_url: https://www.kaggle.com/datasets/mathchi/churn-for-bank-customers
  source_file: /Volumes/bank_churn_eng/bronze/raw_data/churn.csv
      read_at: 2026-08-13T03:42:54+00:00
   size_bytes: 684858
         rows: 10000
      columns: 14
       sha256: 3996cd1fa372e0db0cd9c0ebac35bbd4e8e3c65fb942bb010c826e7b1eeef0a0
       branch: sandbox


## 3 · Eight data quality checks

Each check stores a tuple: **the verdict and the evidence**. Both are computed
independently from the same data, so the evidence prints whether the check passes
or fails. A control that only speaks when something breaks leaves you unable to
tell whether it looked at all.

One honest caveat, worth knowing before anyone asks: **V2 and V4 always pass**.
Their verdict is a hard-coded `True`, because they record shape and ranges rather
than validate them. The ones that genuinely validate are V1, V3, V5, V6, V7 and V8,
where the verdict comes from a comparison.

In [6]:
checks = {}

# V1 - customer_id is unique
checks["V1 customer_id unique"] = (
    raw["CustomerId"].is_unique,
    f"{raw['CustomerId'].nunique():,} distinct out of {len(raw):,} rows",
)

# V2 - shape. Recorded, not validated
checks["V2 dataset shape"] = (
    True, f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")

# V3 - categorical domains
geo = sorted(raw["Geography"].unique())
gen = sorted(raw["Gender"].unique())
checks["V3 categorical domains"] = (
    set(geo) == {"France", "Spain", "Germany"} and set(gen) == {"Male", "Female"},
    f"Geography={geo} | Gender={gen}",
)

# V4 - observed numeric ranges. These are what the CHECK bounds in
# sql/02_modeled.sql were set from. Recorded, not validated
ranges = {c: (raw[c].min(), raw[c].max())
          for c in ["CreditScore", "Age", "Tenure", "Balance",
                    "NumOfProducts", "EstimatedSalary"]}
checks["V4 numeric ranges"] = (
    True, " | ".join(f"{c}[{lo:g}, {hi:g}]" for c, (lo, hi) in ranges.items()))

# V5 - nulls
nulls = raw.isna().sum()
checks["V5 no nulls"] = (nulls.sum() == 0, f"total nulls = {int(nulls.sum())}")

# V6 - division-by-zero risk in the ratio features built in notebook 02
zero_salary = int((raw["EstimatedSalary"] == 0).sum())
age_18      = int((raw["Age"] == 18).sum())
checks["V6 zero-division risk"] = (
    zero_salary == 0 and age_18 == 0,
    f"EstimatedSalary=0 -> {zero_salary} | Age=18 -> {age_18}",
)

# V7 - full-row duplicates, ignoring RowNumber
dups = int(raw.drop(columns=["RowNumber"]).duplicated().sum())
checks["V7 no duplicates"] = (dups == 0, f"{dups} duplicated rows")

# V8 - biographical coherence: you cannot open an account before turning 18,
# so tenure can never exceed age - 18
incoherent = int((raw["Tenure"] > raw["Age"] - 18).sum())
checks["V8 tenure coherent"] = (incoherent == 0, f"{incoherent} incoherent rows")

print(f"{'check':<26} {'':<4} evidence")
print("-" * 104)
for name, (ok, detail) in checks.items():
    print(f"{name:<26} {'PASS  ' if ok else 'REVIEW'}  {detail}")

check                           evidence
--------------------------------------------------------------------------------------------------------
V1 customer_id unique      PASS    10,000 distinct out of 10,000 rows
V2 dataset shape           PASS    10,000 rows x 14 columns
V3 categorical domains     PASS    Geography=['France', 'Germany', 'Spain'] | Gender=['Female', 'Male']
V4 numeric ranges          PASS    CreditScore[350, 850] | Age[18, 92] | Tenure[0, 10] | Balance[0, 250898] | NumOfProducts[1, 4] | EstimatedSalary[11.58, 199992]
V5 no nulls                PASS    total nulls = 0
V6 zero-division risk      REVIEW  EstimatedSalary=0 -> 0 | Age=18 -> 22
V7 no duplicates           PASS    0 duplicated rows
V8 tenure coherent         REVIEW  322 incoherent rows


## 4 · V8 failed. Pulling the thread

`tenure` measures years as a customer and `age` is the current age. Since accounts
cannot be opened before 18, the most someone can have been a customer is
`age - 18`. Rows above that line describe a biography that cannot happen.

What follows is not four separate checks. It is one investigation, where each
question opens the next: how many, how bad, is it a real segment, and what caused
it.

In [7]:
inc = raw[raw["Tenure"] > raw["Age"] - 18]

print(f"incoherent rows: {len(inc):,}  ({len(inc)/len(raw):.1%})")
print("\nage of those customers:")
print(inc["Age"].describe()[["min", "25%", "50%", "max"]].to_string())

# Implied age at account opening. A negative value means an account opened
# before the customer was born, which is not a dubious value but an impossible one.
implied = inc["Age"] - inc["Tenure"]
print(f"\nimplied opening age -- min: {implied.min()}, median: {implied.median()}")
print(f"accounts opened before birth: {(implied < 0).sum()}")

# Is this a real segment, or an artefact? If they behaved differently they would
# need separate treatment. If they behave like everyone else, the problem is not
# in them: it is in how the data was generated.
print(f"\nchurn overall     : {raw['Exited'].mean():.1%}")
print(f"churn incoherent  : {inc['Exited'].mean():.1%}")

incoherent rows: 322  (3.2%)

age of those customers:
min    18.0
25%    21.0
50%    22.0
max    27.0

implied opening age -- min: 8, median: 15.0
accounts opened before birth: 0

churn overall     : 20.4%
churn incoherent  : 7.5%


In [8]:
# The decisive test.
#
# In real data tenure grows with age: older people have banked longer. A flat
# zero only happens if the two columns were drawn independently of each other.
rho = raw["Age"].corr(raw["Tenure"], method="spearman")

print(f"Spearman correlation Age-Tenure: {rho:.4f}")
print()
print("Nothing to clean here. There are no corrupt rows: the generator rolled")
print("one die for age and another for tenure, and some combinations came out")
print("impossible. This is the first of four independent signs that the dataset")
print("is synthetic -- the other three appear in notebook 03.")

Spearman correlation Age-Tenure: -0.0104

Nothing to clean here. There are no corrupt rows: the generator rolled
one die for age and another for tenure, and some combinations came out
impossible. This is the first of four independent signs that the dataset
is synthetic -- the other three appear in notebook 03.


In [9]:
# Cross-check on the group where the artefact concentrates. If the incoherent
# young customers churned at a different rate, they would be a segment. They do not.
young = raw[raw["Age"].between(18, 27)]
coh = young[young["Tenure"] <= young["Age"] - 18]
bad = young[young["Tenure"] >  young["Age"] - 18]

print(f"coherent young    : {coh['Exited'].mean():.1%}  (n={len(coh):,})")
print(f"incoherent young  : {bad['Exited'].mean():.1%}  (n={len(bad):,})")
print("\nSame behaviour. The rows stay: dropping 322 customers to hide a")
print("generator artefact would remove real signal and document nothing.")

coherent young    : 7.0%  (n=698)
incoherent young  : 7.5%  (n=322)

Same behaviour. The rows stay: dropping 322 customers to hide a
generator artefact would remove real signal and document nothing.


## 5 · The column contract

Column names are normalised to `snake_case` because Postgres lower-cases any
unquoted identifier — keeping `CustomerId` would mean quoting it in every query
forever.

The two `assert` lines are the actual contract. If the source ever gains or loses
a column, this notebook stops here with a clear message instead of loading a
misaligned table that fails three notebooks later.

In [10]:
COLUMN_MAP = {
    "RowNumber":       "row_number",
    "CustomerId":      "customer_id",
    "Surname":         "surname",
    "CreditScore":     "credit_score",
    "Geography":       "geography",
    "Gender":          "gender",
    "Age":             "age",
    "Tenure":          "tenure",
    "Balance":         "balance",
    "NumOfProducts":   "num_of_products",
    "HasCrCard":       "has_cr_card",
    "IsActiveMember":  "is_active_member",
    "EstimatedSalary": "estimated_salary",
    "Exited":          "exited",
}

missing = set(COLUMN_MAP) - set(raw.columns)
extra   = set(raw.columns) - set(COLUMN_MAP)
assert not missing, f"expected columns not present in the source: {missing}"
assert not extra,   f"unexpected columns in the source: {extra}"

staged = raw.rename(columns=COLUMN_MAP)
staged["_source_file"] = TRACE["source_file"]
staged["_source_url"]  = TRACE["source_url"]

print(f"contract verified -- {len(COLUMN_MAP)} columns mapped")

contract verified -- 14 columns mapped


## 6 · Load into the landing table

`COPY ... FROM STDIN` instead of ten thousand `INSERT` statements. One round trip
rather than ten thousand, and Postgres parses the stream directly.

`TRUNCATE` before loading makes the cell **idempotent**: running it twice leaves
the same ten thousand rows, not twenty thousand. That matters more than it sounds
in a notebook, where re-running a cell is the normal way to work.

In [11]:
import io
from config import PG_SCHEMA

COLS = list(COLUMN_MAP.values()) + ["_source_file", "_source_url"]

buf = io.StringIO()
staged[COLS].to_csv(buf, index=False, header=False)
buf.seek(0)

with ctx.connect() as conn:
    cur = conn.cursor()
    cur.execute(f"TRUNCATE TABLE {PG_SCHEMA}.customers_raw;")
    cur.execute(
        f'COPY {PG_SCHEMA}.customers_raw ({", ".join(COLS)}) FROM STDIN WITH (FORMAT CSV)',
        stream=buf,
    )
    conn.commit()

loaded = ctx.query(f"SELECT COUNT(*) AS n FROM {PG_SCHEMA}.customers_raw").iloc[0, 0]
print(f"loaded: {loaded:,} rows")
assert loaded == len(raw), "row count does not match the source"

loaded: 10,000 rows


## 7 · First business pattern

The verification query already surfaces something worth carrying forward:
**Germany churns at roughly twice the rate of France and Spain**, with about half
the customers France has.

Notebook 03 explores it properly. Note from here, though, that `Geography` is one
of the two variables this project examines from a **fairness** angle, so this
finding has to be revisited as a question about equity — does the model work
equally well in all three countries? — and not only as a commercial opportunity.

In [12]:
pattern = ctx.query(f"""
    SELECT geography,
           COUNT(*)                                   AS customers,
           SUM(exited)                                AS churned,
           ROUND(100.0 * AVG(exited::numeric), 1)     AS churn_rate_pct
    FROM   {PG_SCHEMA}.customers_raw
    GROUP  BY geography
    ORDER  BY churn_rate_pct DESC
""")
print(pattern.to_string(index=False))

geography  customers  churned churn_rate_pct
  Germany       2509      814           32.4
    Spain       2477      413           16.7
   France       5014      810           16.2


## 8 · The modelled table

`customers` is the operational record, not the analytical layer — that is silver,
in notebook 02. It exists for two reasons: it is the foreign-key target of
`customer_predictions`, and it is the privacy boundary.

**`surname` stops here.** It is in `customers_raw` because the landing table is a
faithful copy, and it is absent from `customers` because the schema enforces the
constraint rather than trusting whoever writes the next query. A column that does
not exist cannot be leaked by accident.

If this cell reports a missing table, `sql/02_modeled.sql` has not been run yet.

In [13]:
table_exists = ctx.query(f"""
    SELECT EXISTS (
        SELECT 1 FROM information_schema.tables
        WHERE table_schema = '{PG_SCHEMA}' AND table_name = 'customers'
    ) AS present
""").iloc[0, 0]

if not table_exists:
    print(f"table {PG_SCHEMA}.customers does not exist yet.")
    print("Run sql/02_modeled.sql in the Lakebase SQL editor, then re-run this cell.")
else:
    with ctx.connect() as conn:
        cur = conn.cursor()
        cur.execute(f"TRUNCATE TABLE {PG_SCHEMA}.customers CASCADE;")
        cur.execute(f"""
            INSERT INTO {PG_SCHEMA}.customers (
                customer_id, geography_id, gender, age, tenure, credit_score,
                balance, estimated_salary, num_of_products,
                has_cr_card, is_active_member, exited
            )
            SELECT r.customer_id, g.geography_id, r.gender, r.age, r.tenure,
                   r.credit_score, r.balance, r.estimated_salary, r.num_of_products,
                   r.has_cr_card = 1, r.is_active_member = 1, r.exited = 1
            FROM   {PG_SCHEMA}.customers_raw r
            JOIN   {PG_SCHEMA}.geographies g ON g.country_name = r.geography
        """)
        conn.commit()

    n = ctx.query(f"SELECT COUNT(*) AS n FROM {PG_SCHEMA}.customers").iloc[0, 0]
    print(f"customers populated: {n:,} rows")
    print("surname was not carried over, by design")

customers populated: 10,000 rows
surname was not carried over, by design


## 9 · The Delta copy

Same data, different engine, different job. Delta gives versions and time travel,
which is what makes an analysis reproducible: if a table is overwritten tomorrow,
`VERSION AS OF` still reaches today's state.

`_extracted_at` is not the same as `_ingested_at`. The first records when the row
reached Delta; the second, when it reached Postgres. Keeping both means the two
sides can be reconciled without guessing.

In [14]:
from pyspark.sql import functions as F

BRONZE_TABLE = f"{UC_CATALOG}.bronze.customers_raw"

bronze_df = spark.createDataFrame(staged[COLS])

(bronze_df
 .withColumn("_extracted_at", F.current_timestamp())
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(BRONZE_TABLE))

spark.sql(f"""
    COMMENT ON TABLE {BRONZE_TABLE} IS
    'Bronze. Immutable copy of the source file. No transformations.
     Rebuild source for silver and gold.'
""")

print("written:", BRONZE_TABLE)
print(f"rows   : {spark.table(BRONZE_TABLE).count():,}")

written: bank_churn_eng.bronze.customers_raw
rows   : 10,000


## 10 · Where this leaves things

Three destinations written, one finding documented, and nothing deleted.

The 322 incoherent rows stay in place. Removing them would have hidden the single
most informative fact about this dataset — that it is synthetic — behind a clean
data quality report. The finding is worth more than the cleanliness.

In [15]:
print("Lakebase")
for t in ("customers_raw", "customers"):
    n = ctx.query(f"SELECT COUNT(*) AS n FROM {PG_SCHEMA}.{t}").iloc[0, 0]
    print(f"  {PG_SCHEMA}.{t:<16} {n:>7,} rows")

print("\nDelta")
print(f"  {BRONZE_TABLE:<34} {spark.table(BRONZE_TABLE).count():>7,} rows")

print(f"\nprovenance sha256: {TRACE['sha256'][:16]}...")
print("\nnext: 02_data_preparation.ipynb")

Lakebase
  bank_churn.customers_raw     10,000 rows
  bank_churn.customers         10,000 rows

Delta
  bank_churn_eng.bronze.customers_raw  10,000 rows

provenance sha256: 3996cd1fa372e0db...

next: 02_data_preparation.ipynb
